In [1]:
import os
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from deconveil.dds import deconveil_fit
from deconveil.inference import Inference
from deconveil.default_inference import DefInference
from deconveil.utils_fit import *
from deconveil.utils_processing import *
from deconveil.utils_plot import *
from deconveil import deconveil_fit
from deconveil.ds import deconveil_stats

#### Load TCGA data

In [5]:
rna_counts = load_test_data(
    modality="rna",
    dataset="tcga_brca",
    debug=False,
)
rna_counts = rna_counts.T

metadata = load_test_data(
    modality="metadata",
    dataset="tcga_brca",
    debug=False,
)

cnv = load_test_data(
    modality="cnv",
    dataset="tcga_brca",
    debug=False,
)
cnv = cnv.T

In [7]:
rna_counts.head()

,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2MP1,A3GALT2,A4GALT,A4GNT,AACS,AACSP1,...,ZSWIM6,ZSWIM9,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
GTEX-13PVQ-1026-SM-5KM3M.1,3502210.0,83739.0,1592.0,17369.0,9842.0,4961.0,377148.0,657.0,156623.0,151.0,...,231243.0,11494.0,41101.0,23345.0,4853.0,93143.0,679.0,97130.0,1819661.0,394464.0
GTEX-18QFQ-0826-SM-718AX.1,5394960.0,155914.0,14511.0,10089.0,11023.0,303.0,278056.0,357.0,385961.0,149.0,...,184259.0,83650.0,123117.0,74837.0,25442.0,320704.0,8603.0,300670.0,1580226.0,918922.0
GTEX-1JN6P-2426-SM-ARL99.1,7136304.0,219925.0,2254.0,12267.0,5955.0,769.0,229205.0,0.0,40360.0,0.0,...,114710.0,22251.0,37669.0,17233.0,4141.0,82340.0,372.0,89483.0,982597.0,339959.0
GTEX-13S86-1226-SM-5S2OA.1,5710693.0,236061.0,932.0,4446.0,12644.0,986.0,468005.0,76.0,177527.0,0.0,...,62143.0,22458.0,73701.0,20962.0,8415.0,111953.0,126.0,94875.0,1060350.0,233669.0
GTEX-132NY-0826-SM-5K7Y7.1,4020951.0,155161.0,6312.0,5744.0,5015.0,76.0,480082.0,1286.0,277306.0,152.0,...,106252.0,28196.0,49946.0,35748.0,6204.0,134309.0,505.0,133562.0,1136263.0,392324.0


In [9]:
metadata.head()

,condition
GTEX-13PVQ-1026-SM-5KM3M.1,A
GTEX-18QFQ-0826-SM-718AX.1,A
GTEX-1JN6P-2426-SM-ARL99.1,A
GTEX-13S86-1226-SM-5S2OA.1,A
GTEX-132NY-0826-SM-5K7Y7.1,A


In [11]:
cnv.head()

,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2MP1,A3GALT2,A4GALT,A4GNT,AACS,AACSP1,...,ZSWIM6,ZSWIM9,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
GTEX-13PVQ-1026-SM-5KM3M.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-18QFQ-0826-SM-718AX.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-1JN6P-2426-SM-ARL99.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-13S86-1226-SM-5S2OA.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-132NY-0826-SM-5K7Y7.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2


#### QC check and filtering

In [13]:
print("Before filtering:", rna_counts.shape, cnv.shape, metadata.shape)
# Reorder cnv to match rna_counts
cnv = cnv.loc[rna_counts.index]
assert (rna_counts.index == metadata.index).all(), "Sample order mismatch between rna_counts and metadata"
assert (cnv.index == rna_counts.index).all(), "Sample order mismatch between cnv and rna_counts"

Before filtering: (400, 17387) (400, 17387) (400, 1)


In [15]:
all_zero_mask = (rna_counts.sum(axis=0) == 0)
print("All-zero genes:", all_zero_mask.sum())
rna_counts = rna_counts.loc[:, ~all_zero_mask]
cnv = cnv.loc[:, ~all_zero_mask]  

All-zero genes: 0


In [17]:
res = filter_low_count_genes(rna_counts, other_dfs=[cnv], min_count=300, min_samples=50)
rna_counts = res["filtered_df"]
cnv = res["other_filtered"][0]
print("After low-count filtering:", rna_counts.shape, cnv.shape)

After low-count filtering: (400, 16252) (400, 16252)


In [19]:
metadata["CNmean"] = cnv.mean(axis=1)

In [21]:
print(metadata.head())

                           condition  CNmean
GTEX-13PVQ-1026-SM-5KM3M.1         A     2.0
GTEX-18QFQ-0826-SM-718AX.1         A     2.0
GTEX-1JN6P-2426-SM-ARL99.1         A     2.0
GTEX-13S86-1226-SM-5S2OA.1         A     2.0
GTEX-132NY-0826-SM-5K7Y7.1         A     2.0


In [23]:
metadata["CNstatus"] = pd.cut(
    metadata["CNmean"],
    bins=[-np.inf, 1.8, 2.2, np.inf],
    labels=["Loss", "Neutral", "Gain"]
)
print(metadata.head())

                           condition  CNmean CNstatus
GTEX-13PVQ-1026-SM-5KM3M.1         A     2.0  Neutral
GTEX-18QFQ-0826-SM-718AX.1         A     2.0  Neutral
GTEX-1JN6P-2426-SM-ARL99.1         A     2.0  Neutral
GTEX-13S86-1226-SM-5S2OA.1         A     2.0  Neutral
GTEX-132NY-0826-SM-5K7Y7.1         A     2.0  Neutral


In [29]:
cn_states = ["Neutral", "Gain"]
metadata["CNstatus"] = np.random.choice(cn_states, size=len(metadata))
#metadata = metadata.drop(columns=["CNmean"])
print(metadata.head())

                           condition CNstatus
GTEX-13PVQ-1026-SM-5KM3M.1         A     Gain
GTEX-18QFQ-0826-SM-718AX.1         A  Neutral
GTEX-1JN6P-2426-SM-ARL99.1         A     Gain
GTEX-13S86-1226-SM-5S2OA.1         A     Gain
GTEX-132NY-0826-SM-5K7Y7.1         A  Neutral


In [31]:
metadata["condition"] = metadata["condition"].astype("category")
metadata["CNstatus"] = metadata["CNstatus"].astype("category")

In [33]:
# Full rank design matrix check
from formulaic import model_matrix

X = model_matrix("~ condition + CNstatus", metadata)
np.linalg.matrix_rank(X.values), X.shape

(3, (400, 3))

#### Test DeConveil multifactor design

In [35]:
inference = DefInference(n_cpus=8)
dds = deconveil_fit(
        counts=rna_counts,
        cnv=cnv,
        metadata=metadata,
        design="~CNstatus",
        design_factors=None,
        inference=inference,
        refit_cooks=True
    )

In [37]:
np.linalg.matrix_rank(dds.obsm["design_matrix"].values)

2

In [39]:
print(dds.formulaic_contrasts.design_matrix.head())

                            Intercept  CNstatus[T.Neutral]
GTEX-13PVQ-1026-SM-5KM3M.1        1.0                    0
GTEX-18QFQ-0826-SM-718AX.1        1.0                    1
GTEX-1JN6P-2426-SM-ARL99.1        1.0                    0
GTEX-13S86-1226-SM-5S2OA.1        1.0                    0
GTEX-132NY-0826-SM-5K7Y7.1        1.0                    1


In [41]:
#Quick sanity checks
X = dds.obsm["design_matrix"]
print(X.columns.tolist())
print("rank:", np.linalg.matrix_rank(X.values), " / ncols:", X.shape[1])
print(metadata["condition"].value_counts())
print(metadata["CNstatus"].value_counts())

['Intercept', 'CNstatus[T.Neutral]']
rank: 2  / ncols: 2
condition
A    200
B    200
Name: count, dtype: int64
CNstatus
Gain       214
Neutral    186
Name: count, dtype: int64


In [43]:
# Fit coefficients 

dds.fit_size_factors()
dds.fit_genewise_dispersions()
dds.fit_dispersion_trend()
dds.fit_dispersion_prior()
dds.fit_MAP_dispersions()
dds.fit_LFC()
dds.calculate_cooks()

if dds.refit_cooks:
    dds.refit()  # Replace outlier counts

Fitting size factors...


Using None as control genes, passed at deconveil_fit initialization


... done in 0.22 seconds.

Fitting dispersions...
... done in 4.89 seconds.

Fitting dispersion trend curve...
/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/dds.py:1023: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  1 / self.varm["_normed_means"][self.non_zero_idx],
... done in 0.34 seconds.

Fitting MAP dispersions...
... done in 5.45 seconds.

Fitting LFCs...
... done in 4.47 seconds.

Calculating cook's distance...
... done in 0.80 seconds.

Replacing 2605 outlier genes.



replace_mask before filtering: (400, 2605)
Number of True values in replace_mask: 8965
replacement_counts_trimmed shape: (371, 2605)


Fitting dispersions...
... done in 0.46 seconds.

Fitting MAP dispersions...
/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/dds.py:798: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alpha_hat=self.varm["fitted_dispersions"][self.non_zero_idx],
/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/default_inference.py:156: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alpha_hat=alpha_hat[i],
... done in 0.88 seconds.

Fitting LFCs...
... done in 0.46 seconds.



In [45]:
list(dds.varm.keys())

['_normed_means',
 'non_zero',
 '_MoM_dispersions',
 'genewise_dispersions',
 '_genewise_converged',
 'fitted_dispersions',
 'MAP_dispersions',
 '_MAP_converged',
 'dispersions',
 '_outlier_genes',
 'LFC',
 '_LFC_converged',
 'replaced',
 'refitted']

In [47]:
# Stat DE test

stat_res_deconveil = deconveil_stats(
    dds, 
    alpha=0.05, 
    contrast=["CNstatus", "Neutral", "Gain"],
    independent_filter=True, 
    cooks_filter=True
)

stat_res_deconveil.run_wald_test()

if stat_res_deconveil.independent_filter:
    stat_res_deconveil._independent_filtering()
else:
    stat_res_deconveil._p_value_adjustment()

# Log-fold change shrinkage
stat_res_deconveil.lfc_shrink(coeff="CNstatus[T.Neutral]")
stat_res_deconveil.summary()

Running Wald tests...
... done in 0.71 seconds.

Fitting MAP LFCs...


Log2 fold change & Wald test p-value: CNstatus Neutral vs Gain
               baseMean  log2FoldChange     lfcSE      stat    pvalue  \
A2M        5.180468e+06        1.756279  0.023401 -0.126690  0.899185   
A2M-AS1    1.578345e+05        1.353253  0.101611 -0.770976  0.440721   
A2ML1      4.207932e+03       -0.917683  0.023810  2.534416  0.011263   
A2ML1-AS1  7.217329e+03        0.060571  0.108578  1.598566  0.109917   
A2MP1      7.010010e+03       -0.073811  0.039733  2.659318  0.007830   
...                 ...             ...       ...       ...       ...   
ZXDC       2.017188e+05        1.573793  0.045705  2.804523  0.005039   
ZYG11A     2.195483e+04        1.098919  0.208229 -1.186801  0.235306   
ZYG11B     2.067749e+05        1.541374  0.046642  0.621868  0.534029   
ZYX        1.429673e+06        1.755542  0.037172  0.430613  0.666750   
ZZEF1      4.326109e+05        1.664787  0.052721  1.918625  0.055032   

               padj  
A2M        0.956510  
A2M-AS1    0.669

... done in 3.79 seconds.

/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/ds.py:430: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.LFC.iloc[:, coeff_idx].update(
/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/ds.py:450: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ True  True  True ...  True  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  self._LF